To build an automated ETL pipeline that extracts data from multiple sources, transforms it
into a standardized format, and loads the processed data for storage.


Aim

To build an automated ETL (Extract, Transform, Load) pipeline that extracts data from multiple sources, transforms it into a clean and standardized format, and loads the processed data for storage and further analysis.

## Theory

ETL stands for **Extract, Transform, and Load**. It is a common process used to collect data from different sources, clean and organize it, and store it in a suitable format for analysis.

### 1. Extract

In the extraction stage, data is collected from one or more sources. In this experiment, data was extracted from two public REST APIs using Python's `requests` library. The API responses were received in JSON format.

### 2. Transform

The extracted JSON data was converted into Pandas DataFrames. The data was then cleaned and standardized by:

* Standardizing column names
* Removing duplicate records
* Selecting relevant columns
* Renaming columns into a consistent format
* Checking missing values
* Combining the user and post datasets using `user_id`

### 3. Load

In the loading stage, the transformed data was stored as CSV files. The final dataset and city-wise summary were saved for future analysis.

Thus, the ETL pipeline converts raw data from multiple sources into structured and usable information.


In [ ]:
# Install required libraries
!pip install requests pandas

In [ ]:
import requests
import pandas as pd

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# EXTRACT: Fetch data from two public API sources

users_url = "https://jsonplaceholder.typicode.com/users"
posts_url = "https://jsonplaceholder.typicode.com/posts"

users_response = requests.get(users_url)
posts_response = requests.get(posts_url)

print("Users API Status:", users_response.status_code)
print("Posts API Status:", posts_response.status_code)

# Convert API responses to JSON
users_data = users_response.json()
posts_data = posts_response.json()

print("\nUsers records extracted:", len(users_data))
print("Posts records extracted:", len(posts_data))

Users API Status: 200
Posts API Status: 200

Users records extracted: 10
Posts records extracted: 100


In [ ]:
# Convert extracted JSON data into DataFrames

users_df = pd.json_normalize(users_data)
posts_df = pd.json_normalize(posts_data)

print("Users Dataset:")
display(users_df.head())

print("\nPosts Dataset:")
display(posts_df.head())

Users Dataset:


,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,revolutionize end-to-end systems



Posts Dataset:


,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...


In [ ]:
# TRANSFORM: Clean and standardize the datasets

# Standardize column names
users_df.columns = users_df.columns.str.replace(".", "_", regex=False).str.lower()
posts_df.columns = posts_df.columns.str.lower()

# Remove duplicate records
users_df = users_df.drop_duplicates()
posts_df = posts_df.drop_duplicates()

# Select useful user columns
users_clean = users_df[
    ["id", "name", "username", "email", "address_city", "company_name"]
].copy()

# Rename columns for a standard format
users_clean = users_clean.rename(columns={
    "id": "user_id",
    "address_city": "city",
    "company_name": "company"
})

# Select useful post columns
posts_clean = posts_df[
    ["userid", "id", "title", "body"]
].copy()

posts_clean = posts_clean.rename(columns={
    "userid": "user_id",
    "id": "post_id"
})

print("Users after transformation:", users_clean.shape)
print("Posts after transformation:", posts_clean.shape)

print("\nClean Users Data:")
display(users_clean.head())

print("\nClean Posts Data:")
display(posts_clean.head())

Users after transformation: (10, 6)
Posts after transformation: (100, 4)

Clean Users Data:


,user_id,name,username,email,city,company
0,1,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,Romaguera-Crona
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,Wisokyburgh,Deckow-Crist
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,McKenziehaven,Romaguera-Jacobson
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,South Elvis,Robel-Corkery
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,Roscoeview,Keebler LLC



Clean Posts Data:


,user_id,post_id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...


In [ ]:
# Data validation after transformation

print("=== USERS DATA VALIDATION ===")
print("Missing values:")
print(users_clean.isnull().sum())

print("\nDuplicate rows:", users_clean.duplicated().sum())

print("\n=== POSTS DATA VALIDATION ===")
print("Missing values:")
print(posts_clean.isnull().sum())

print("\nDuplicate rows:", posts_clean.duplicated().sum())

=== USERS DATA VALIDATION ===
Missing values:
user_id     0
name        0
username    0
email       0
city        0
company     0
dtype: int64

Duplicate rows: 0

=== POSTS DATA VALIDATION ===
Missing values:
user_id    0
post_id    0
title      0
body       0
dtype: int64

Duplicate rows: 0


In [ ]:
# Combine users and posts using user_id

final_df = posts_clean.merge(
    users_clean,
    on="user_id",
    how="left"
)

print("Final combined dataset shape:", final_df.shape)

display(final_df.head(10))

Final combined dataset shape: (100, 9)


,user_id,post_id,title,body,name,username,email,city,company
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,Romaguera-Crona
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,Romaguera-Crona
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,Romaguera-Crona
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,Romaguera-Crona
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,Romaguera-Crona
5,1,6,dolorem eum magni eos aperiam quia,ut aspernatur corporis harum nihil quis provid...,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,Romaguera-Crona
6,1,7,magnam facilis autem,dolore placeat quibusdam ea quo vitae\nmagni q...,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,Romaguera-Crona
7,1,8,dolorem dolore est ipsam,dignissimos aperiam dolorem qui eum\nfacilis q...,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,Romaguera-Crona
8,1,9,nesciunt iure omnis dolorem tempora et accusan...,consectetur animi nesciunt iure dolore\nenim q...,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,Romaguera-Crona
9,1,10,optio molestias id quia eum,quo et expedita modi cum officia vel magni\ndo...,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,Romaguera-Crona


In [ ]:
# Create a simple summary for analysis

summary = final_df.groupby("city").agg(
    total_posts=("post_id", "count"),
    unique_users=("user_id", "nunique")
).reset_index()

summary = summary.sort_values("total_posts", ascending=False)

print("Posts by City:")
display(summary)

Posts by City:


,city,total_posts,unique_users
0,Aliyaview,10,1
1,Bartholomebury,10,1
2,Gwenborough,10,1
3,Howemouth,10,1
4,Lebsackbury,10,1
5,McKenziehaven,10,1
6,Roscoeview,10,1
7,South Christy,10,1
8,South Elvis,10,1
9,Wisokyburgh,10,1


In [ ]:
# LOAD: Save processed data for future analysis

final_df.to_csv("etl_final_data.csv", index=False)
summary.to_csv("etl_city_summary.csv", index=False)

print("ETL PIPELINE COMPLETED SUCCESSFULLY!")
print("\nFiles created:")
print("1. etl_final_data.csv")
print("2. etl_city_summary.csv")

ETL PIPELINE COMPLETED SUCCESSFULLY!

Files created:
1. etl_final_data.csv
2. etl_city_summary.csv


## Conclusion

The ETL pipeline was successfully implemented using Python. Data was extracted from multiple REST APIs, transformed by cleaning, standardizing and merging the datasets, and finally loaded into CSV files. The experiment demonstrated how ETL can automate the process of preparing data from different sources for storage and further analysis.
